# 01 | Churn Prediction

Train a weighted XGBoost–LightGBM ensemble on the February cohort and score the March cohort. This selected implementation starts with pre-engineered tables; see [data requirements](../data/README.md). It has not been rerun for this portfolio edition.

The historical feature cutoff dates are January 31 and February 28, 2017. Cross-validation is stratified within the training cohort, not a second temporal holdout.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# Run from the repository root or notebooks/ directory.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"
PARQ = DATA / "parquet"
OUTPUT = ROOT / "outputs"
OUTPUT.mkdir(exist_ok=True)
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import log_loss
from scipy.stats import norm
import matplotlib.pyplot as plt

## Load pre-engineered features and member attributes

In [ ]:
def load_members(cutoff):

    m = pd.read_parquet(f'{PARQ}/members_v3.parquet',
                        columns=['msno', 'city', 'bd', 'gender', 'registered_via', 'registration_init_time'])
    m['msno'] = m['msno'].astype(str)
    m['bd'] = np.where((m['bd'] < 1) | (m['bd'] > 100), np.nan, m['bd'])
    m['gender_enc'] = m['gender'].map({'male': 1, 'female': 0})
    cut = pd.to_datetime(str(cutoff), format='%Y%m%d')
    reg = pd.to_datetime(m['registration_init_time'].astype(str), format='%Y%m%d', errors='coerce')
    m['tenure_days'] = (cut - reg).dt.days
    return m[['msno', 'city', 'bd', 'gender_enc', 'registered_via', 'tenure_days']]

def load_cohort(top_pq, cutoff, label_pq=None):
    f = pd.read_parquet(top_pq); f['msno'] = f['msno'].astype(str)
    d = f.merge(load_members(cutoff), on='msno', how='left')
    if label_pq:
        lab = pd.read_parquet(label_pq, columns=['msno', 'is_churn']); lab['msno'] = lab['msno'].astype(str)
        d = d.merge(lab, on='msno', how='inner')
    return f, d

feats, d = load_cohort(f'{PARQ}/top_features_train.parquet', 20170131, f'{PARQ}/train_v2.parquet')
feat_cols = [c for c in d.columns if c not in ('msno', 'is_churn')]
Xtr = d[feat_cols].astype('float32').values
ytr = d['is_churn'].values
print(f'Training matrix: {Xtr.shape}, churn={ytr.mean():.4f}')

## Train and calibrate

Five-fold predictions are used to choose ensemble weights and fit Platt scaling. The calibration score is computed on those same OOF predictions, so it is not an independent estimate of calibration performance. The original XGBoost CV prediction uses all trees returned by early stopping; the full-data refit uses the last fold’s best iteration for each learner. These choices are retained for historical fidelity.

In [ ]:
XGBP = dict(max_depth=8, learning_rate=0.02, subsample=0.8, colsample_bytree=0.7, min_child_weight=10,
            reg_lambda=1.5, reg_alpha=0.5, objective='binary:logistic', eval_metric='logloss', tree_method='hist', nthread=6)
LGBP = dict(num_leaves=127, learning_rate=0.02, subsample=0.8, subsample_freq=1, colsample_bytree=0.7,
            min_child_samples=100, reg_lambda=1.5, objective='binary', n_jobs=6, verbose=-1)

skf = StratifiedKFold(5, shuffle=True, random_state=42)
oof_xgb = np.zeros(len(ytr)); oof_lgb = np.zeros(len(ytr))
for k, (tr, va) in enumerate(skf.split(Xtr, ytr)):
    dtr = xgb.DMatrix(Xtr[tr], label=ytr[tr], feature_names=feat_cols)
    dva = xgb.DMatrix(Xtr[va], label=ytr[va], feature_names=feat_cols)
    bst = xgb.train(XGBP, dtr, 6000, evals=[(dva, 'v')], early_stopping_rounds=120, verbose_eval=False)
    oof_xgb[va] = bst.predict(dva)
    lgm = lgb.LGBMClassifier(n_estimators=6000, **LGBP)
    lgm.fit(Xtr[tr], ytr[tr], eval_set=[(Xtr[va], ytr[va])], callbacks=[lgb.early_stopping(120, verbose=False)])
    oof_lgb[va] = lgm.predict_proba(Xtr[va])[:, 1]
    print(f'  fold {k}: xgb={log_loss(ytr[va], oof_xgb[va]):.5f}  lgb={log_loss(ytr[va], oof_lgb[va]):.5f}')


best_w, best_ll = 0.5, 9
for w in np.arange(0, 1.01, 0.05):
    ll = log_loss(ytr, w * oof_xgb + (1 - w) * oof_lgb)
    if ll < best_ll: best_ll, best_w = ll, w

oof_ens = best_w * oof_xgb + (1 - best_w) * oof_lgb
oof_logit = np.log(np.clip(oof_ens, 1e-6, 1 - 1e-6) / (1 - np.clip(oof_ens, 1e-6, 1 - 1e-6)))
platt = LogisticRegression(C=1e6, max_iter=1000).fit(oof_logit.reshape(-1, 1), ytr)
print(f'\nEnsemble OOF log loss={best_ll:.5f} (w_xgb={best_w});after Platt scaling={log_loss(ytr, platt.predict_proba(oof_logit.reshape(-1,1))[:,1]):.5f}')


xgb_full = xgb.train(XGBP, xgb.DMatrix(Xtr, label=ytr, feature_names=feat_cols), num_boost_round=int(bst.best_iteration * 1.1) + 50)
lgb_full = lgb.LGBMClassifier(n_estimators=int(lgm.best_iteration_ * 1.1) + 50, **LGBP).fit(Xtr, ytr)
print('Full-data refit complete')

## Score the later cohort and explore base-rate adjustment

The 3.5% target was selected using leaderboard feedback. The values plotted below are historical submission results recorded in the report, not newly computed scores or an untouched test evaluation.

In [ ]:
_, dte = load_cohort(f'{PARQ}/top_features_test.parquet', 20170228)
Xte = dte[feat_cols].astype('float32').values
p_te = best_w * xgb_full.predict(xgb.DMatrix(Xte, feature_names=feat_cols)) + (1 - best_w) * lgb_full.predict_proba(Xte)[:, 1]
lg_te = np.log(np.clip(p_te, 1e-6, 1 - 1e-6) / (1 - np.clip(p_te, 1e-6, 1 - 1e-6)))
p_te_cal = platt.predict_proba(lg_te.reshape(-1, 1))[:, 1]


def shift_to_mean(logit, target):
    lo, hi = -10, 10
    for _ in range(60):
        mid = (lo + hi) / 2
        if (1 / (1 + np.exp(-(logit - mid)))).mean() > target: lo = mid
        else: hi = mid
    return 1 / (1 + np.exp(-(logit - (lo + hi) / 2)))
p_final = shift_to_mean(np.log(np.clip(p_te_cal, 1e-6, 1-1e-6)/(1-np.clip(p_te_cal,1e-6,1-1e-6))), 0.035)
print(f'Test mean after Platt scaling: {p_te_cal.mean():.4f} → adjusted to {p_final.mean():.4f}')


allm = pd.read_parquet(f'{PARQ}/sample_submission_v2.parquet', columns=['msno']); allm['msno'] = allm['msno'].astype(str)
sub = allm.merge(pd.DataFrame({'msno': dte['msno'].astype(str).values, 'is_churn': p_final}), on='msno', how='left')
sub['is_churn'] = sub['is_churn'].fillna(0.035).clip(1e-6, 1-1e-6)
# Keep individual predictions local; export only if required.
# sub.to_csv(OUTPUT / 'submission_full.csv', index=False)
print(f'Submission rows ({len(sub):,} rows)')


means=[0.025,0.030,0.035,0.040,0.064]; lls=[0.12834,0.12666,0.12611,0.12642,0.13542]
plt.figure(figsize=(6,3.4)); plt.plot(means,lls,'-o'); plt.scatter([0.035],[0.12611],s=140,c='g',zorder=5)
plt.annotate('Best reported: 0.12611',(0.035,0.12611),xytext=(8,10),textcoords='offset points',color='g')
plt.xlabel('Mean predicted churn probability'); plt.ylabel('Private Log Loss'); plt.title('Historical leaderboard feedback'); plt.tight_layout(); plt.show()

## Historical results (recorded, not rerun)

| Metric | Value |
|---|---:|
| Training rows | 937,817 |
| Features | 78 |
| Weighted ensemble OOF log loss | 0.17406 |
| OOF log loss after fitted Platt scaling | 0.17400 |
| Private leaderboard before base-rate adjustment | 0.13542 |
| Private leaderboard after adjustment | 0.12611 |

## Prepare the local experiment-design input

The next cell reproduces the original full-model scoring of training rows (in-sample). It saves a local, non-public intermediate for notebook 04. An independent validation cohort or cross-fitted scores would be preferable for prospective risk estimates.

In [ ]:
Xm=d[feat_cols].astype('float32').values
pm=best_w*xgb_full.predict(xgb.DMatrix(Xm,feature_names=feat_cols))+(1-best_w)*lgb_full.predict_proba(Xm)[:,1]
lgm2=np.log(np.clip(pm,1e-6,1-1e-6)/(1-np.clip(pm,1e-6,1-1e-6))); d['risk']=platt.predict_proba(lgm2.reshape(-1,1))[:,1]

# Local intermediate for notebook 04; no identifiers are exported.
d[['risk', 'last_ul_days_s', 'last_price', 'is_churn']].to_parquet(OUTPUT / 'experiment_risk_input.parquet', index=False)